# LeetCode 328: Odd Even Linked List

**Difficulty**: Medium  
**Topics**: Linked List, Two Pointers  
**Link**: [LeetCode Problem](https://leetcode.com/problems/odd-even-linked-list/)

---

## Problem Statement

Given the `head` of a singly linked list, group all the nodes with **odd indices** together followed by the nodes with **even indices**, and return the reordered list.

The **first** node is considered **odd** (index 0), and the **second** node is **even** (index 1), and so on.

**Note**: The relative order inside both the even and odd groups should remain as it was in the input.

You must solve the problem in **O(1) extra space** complexity and **O(n) time** complexity.

### Visual Example

```
Input:  1 -> 2 -> 3 -> 4 -> 5
        ^    ^    ^    ^    ^
      odd  even odd  even odd

Output: 1 -> 3 -> 5 -> 2 -> 4
        └─odd nodes─┘  └even─┘
```

### Examples

**Example 1:**
```
Input: head = [1,2,3,4,5]
Output: [1,3,5,2,4]
```

**Example 2:**
```
Input: head = [2,1,3,5,6,4,7]
Output: [2,3,6,7,1,5,4]
```

### Constraints

- The number of nodes is in the range `[0, 10^4]`
- `-10^6 <= Node.val <= 10^6`

---

## Approach 1: Create New Lists (Extra Space)

### Intuition

Create two separate lists: one for odd-indexed nodes and one for even-indexed nodes. Then concatenate them.

### Algorithm

```
1. Traverse the original list
2. Add odd-indexed nodes to odd list
3. Add even-indexed nodes to even list
4. Connect odd list tail to even list head
```

### Complexity

- **Time**: O(n)
- **Space**: O(n) - creates new nodes

**Note**: This violates the O(1) space requirement, so it's not optimal.

In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def oddEvenList_extraSpace(head):
    """
    Creates new lists (violates O(1) space requirement).
    Time: O(n), Space: O(n)
    """
    if not head:
        return None
    
    odd_dummy = ListNode(0)
    even_dummy = ListNode(0)
    odd_tail = odd_dummy
    even_tail = even_dummy
    
    index = 0
    current = head
    
    while current:
        if index % 2 == 0:  # Odd position (0-indexed)
            odd_tail.next = ListNode(current.val)
            odd_tail = odd_tail.next
        else:  # Even position
            even_tail.next = ListNode(current.val)
            even_tail = even_tail.next
        current = current.next
        index += 1
    
    # Connect odd list to even list
    odd_tail.next = even_dummy.next
    return odd_dummy.next

# Helper function to create list from array
def create_list(arr):
    if not arr:
        return None
    head = ListNode(arr[0])
    current = head
    for val in arr[1:]:
        current.next = ListNode(val)
        current = current.next
    return head

# Helper function to convert list to array
def list_to_array(head):
    result = []
    while head:
        result.append(head.val)
        head = head.next
    return result

# Test
head = create_list([1, 2, 3, 4, 5])
result = oddEvenList_extraSpace(head)
print(f"Input:  [1, 2, 3, 4, 5]")
print(f"Output: {list_to_array(result)}")
print(f"Expected: [1, 3, 5, 2, 4]")

---

## Approach 2: In-Place Rearrangement (Optimal)

### Intuition

Instead of creating new nodes, **rearrange pointers** in the existing list.

**Key insight**: Maintain two separate chains (odd and even) by rewiring pointers, then connect them at the end.

### Visual Walkthrough

For `[1, 2, 3, 4, 5]`:

```
Initial:
1 -> 2 -> 3 -> 4 -> 5 -> null
^    ^
odd  even

Step 1: odd.next = even.next (skip 2, point to 3)
1 ───────> 3 -> 4 -> 5 -> null
     2 ───^
     
Step 2: odd = odd.next (move odd to 3)
1 -> 3 -> 4 -> 5 -> null
     ^    
    odd
    
Step 3: even.next = odd.next (skip 3, point to 4)
2 ───────> 4 -> 5 -> null
     
Step 4: even = even.next (move even to 4)
2 -> 4 -> 5 -> null
     ^
    even
    
Continue until even.next is null...

Final odd chain:  1 -> 3 -> 5
Final even chain: 2 -> 4

Connect: odd.next = even_head
Result: 1 -> 3 -> 5 -> 2 -> 4
```

### Algorithm

```
1. If list is empty, return null
2. Initialize:
     odd = head (first node)
     even = head.next (second node)
     even_head = even (save even list start)
3. While even and even.next exist:
     odd.next = even.next    # Skip even node
     odd = odd.next          # Move odd pointer
     even.next = odd.next    # Skip odd node
     even = even.next        # Move even pointer
4. Connect: odd.next = even_head
5. Return head
```

### Why This Works

- **Odd pointer** always points to an odd-indexed node
- **Even pointer** always points to an even-indexed node
- By skipping one node at a time, we build two separate chains
- We save `even_head` to reconnect the chains at the end

### Complexity

- **Time**: O(n) - single pass through the list
- **Space**: O(1) - only a few pointers, no new nodes

In [ ]:
def oddEvenList(head):
    """
    Optimal in-place solution.
    Time: O(n), Space: O(1)
    """
    if not head or not head.next:
        return head
    
    odd = head
    even = head.next
    even_head = even  # Save the start of even list
    
    # Build odd and even chains simultaneously
    while even and even.next:
        odd.next = even.next    # Odd skips even node
        odd = odd.next          # Move odd pointer forward
        even.next = odd.next    # Even skips odd node
        even = even.next        # Move even pointer forward
    
    # Connect odd chain to even chain
    odd.next = even_head
    return head

# Test cases
test_cases = [
    [1, 2, 3, 4, 5],
    [2, 1, 3, 5, 6, 4, 7],
    [1],
    [1, 2],
]

for arr in test_cases:
    head = create_list(arr)
    result = oddEvenList(head)
    print(f"Input:  {arr}")
    print(f"Output: {list_to_array(result)}\n")

### Detailed Step-by-Step Trace

Let's trace through `[1, 2, 3, 4, 5]` with detailed pointer states:

In [ ]:
def oddEvenList_verbose(head):
    """Verbose version showing each step."""
    if not head or not head.next:
        return head
    
    print("Initial list:", list_to_array(head))
    print()
    
    odd = head
    even = head.next
    even_head = even
    
    print(f"odd = {odd.val}, even = {even.val}, even_head = {even_head.val}\n")
    
    step = 1
    while even and even.next:
        print(f"Step {step}:")
        print(f"  Before: odd={odd.val}, even={even.val}")
        
        # Odd skips even
        odd.next = even.next
        print(f"  odd.next = even.next → {odd.val}.next = {odd.next.val}")
        
        # Move odd
        odd = odd.next
        print(f"  odd moves to {odd.val}")
        
        # Even skips odd
        even.next = odd.next
        if odd.next:
            print(f"  even.next = odd.next → {even.val}.next = {even.next.val}")
        else:
            print(f"  even.next = odd.next → {even.val}.next = null")
        
        # Move even
        even = even.next
        if even:
            print(f"  even moves to {even.val}")
        else:
            print(f"  even moves to null")
        
        print(f"  After: odd={odd.val}, even={even.val if even else 'null'}\n")
        step += 1
    
    print(f"Loop ended. Connecting odd chain to even chain...")
    print(f"odd.next = even_head → {odd.val}.next = {even_head.val}\n")
    
    odd.next = even_head
    
    print("Final result:", list_to_array(head))
    return head

# Test
head = create_list([1, 2, 3, 4, 5])
oddEvenList_verbose(head)

---

## Edge Cases

In [ ]:
# Edge case 1: Empty list
print("Empty list:")
result = oddEvenList(None)
print(f"Output: {list_to_array(result)}\n")

# Edge case 2: Single node
print("Single node:")
head = create_list([1])
result = oddEvenList(head)
print(f"Input:  [1]")
print(f"Output: {list_to_array(result)}\n")

# Edge case 3: Two nodes
print("Two nodes:")
head = create_list([1, 2])
result = oddEvenList(head)
print(f"Input:  [1, 2]")
print(f"Output: {list_to_array(result)}\n")

# Edge case 4: Even number of nodes
print("Even number of nodes:")
head = create_list([1, 2, 3, 4])
result = oddEvenList(head)
print(f"Input:  [1, 2, 3, 4]")
print(f"Output: {list_to_array(result)}")
print(f"Expected: [1, 3, 2, 4]\n")

# Edge case 5: Odd number of nodes
print("Odd number of nodes:")
head = create_list([1, 2, 3, 4, 5])
result = oddEvenList(head)
print(f"Input:  [1, 2, 3, 4, 5]")
print(f"Output: {list_to_array(result)}")
print(f"Expected: [1, 3, 5, 2, 4]")

---

## Common Mistakes

### Mistake 1: Not Saving even_head

**Wrong**:
```python
odd = head
even = head.next
# Forgot to save even_head!

while even and even.next:
    # ... rearrange ...

odd.next = even  # Wrong! even has moved
```

**Right**:
```python
odd = head
even = head.next
even_head = even  # Save the start of even list

# ... rearrange ...

odd.next = even_head  # Correct! Connect to saved head
```

### Mistake 2: Wrong Loop Condition

**Wrong**:
```python
while odd and odd.next:  # Wrong pointer
```

**Right**:
```python
while even and even.next:  # Check even pointer
```

We check `even` because it's always ahead of `odd`. If `even.next` exists, we can safely do `odd.next = even.next`.

### Mistake 3: Wrong Pointer Update Order

**Wrong**:
```python
odd = odd.next          # Move first
odd.next = even.next    # Then update - but odd has moved!
```

**Right**:
```python
odd.next = even.next    # Update first
odd = odd.next          # Then move
```

### Mistake 4: Forgetting Edge Cases

Always check:
- Empty list (`head == None`)
- Single node (`head.next == None`)
- Two nodes (minimal odd/even split)

---

## Comparison of Approaches

| Approach | Time | Space | Meets Requirements? |
|----------|------|-------|--------------------|
| Create New Lists | O(n) | O(n) | ❌ No (violates O(1) space) |
| In-Place Rearrangement | O(n) | O(1) | ✅ Yes (optimal) |

The in-place approach is the only solution that meets both the time and space requirements.

---

## Related Problems

- [86. Partition List](https://leetcode.com/problems/partition-list/) - Similar two-pointer rearrangement
- [2. Add Two Numbers](https://leetcode.com/problems/add-two-numbers/) - Linked list manipulation
- [24. Swap Nodes in Pairs](https://leetcode.com/problems/swap-nodes-in-pairs/) - Pointer rearrangement pattern
- [143. Reorder List](https://leetcode.com/problems/reorder-list/) - More complex rearrangement

---

## Key Takeaways

1. **Odd/even** refers to **node position** (index), not node value
2. **Save even_head** before starting the rearrangement
3. **Two pointers** (odd and even) build separate chains simultaneously
4. **Loop condition**: `while even and even.next` (even is always ahead)
5. **Update order matters**: Update pointers before moving them
6. **O(1) space** achieved by rewiring existing pointers, not creating new nodes
7. **Pattern**: Separate chains by property, then reconnect
8. **Edge cases**: Handle empty, single node, and two-node lists

### The Core Insight

Instead of creating new lists, we **rewire the existing nodes** into two chains (odd and even), then connect them. This achieves O(1) space complexity while maintaining O(n) time complexity.